# Seq2Seq 모델

In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.7/472.7 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 19.2 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

dataset = load_dataset("wmt14", "de-en")

print(dataset['train'][0])

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/265M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/273M [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/509k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

{'translation': {'de': 'Wiederaufnahme der Sitzungsperiode', 'en': 'Resumption of the session'}}


In [ ]:
temp = dataset["train"]["translation"]
for i in temp:
    print(i)
    break

{'de': 'Wiederaufnahme der Sitzungsperiode', 'en': 'Resumption of the session'}


In [ ]:
with open("de_corpus.txt", "w") as de_corpus, open("en_corpus.txt", "w") as en_corpus:
    for pair in temp:
        de_corpus.write(pair["de"] + "\n")
        en_corpus.write(pair["en"] + "\n")

In [ ]:
import sentencepiece as spm

vocab_size = 8000
pad_id = 0
bos_id = 1
eos_id = 2
unk_id = 3

spm.SentencePieceTrainer.train(
    input = "de_corpus.txt",
    model_prefix = "encoder_spm",
    vocab_size = vocab_size,
    pad_id = pad_id,
    bos_id = bos_id,
    eos_id = eos_id,
    unk_id = unk_id
)

spm.SentencePieceTrainer.train(
    input = "en_corpus.txt",
    model_prefix = "decoder_spm",
    vocab_size = vocab_size,
    pad_id = pad_id,
    bos_id = bos_id,
    eos_id = eos_id,
    unk_id = unk_id
)

encoder_tokenizer = spm.SentencePieceProcessor()
encoder_tokenizer.load("encoder_spm.model")

decoder_tokenizer = spm.SentencePieceProcessor()
decoder_tokenizer.load("decoder_spm.model")

sample_de_text = "Wiederaufnahme der Sitzungsperiode"
sample_en_text = "Resumption of the session"

# 독일어 및 영어 문장을 토큰화
de_ids = encoder_tokenizer.encode(sample_de_text, out_type=int)
en_ids = decoder_tokenizer.encode(sample_en_text, out_type=int)

print("독일어 토큰화:", de_ids)
print("영어 토큰화:", en_ids)
print("독일어 텍스트:", encoder_tokenizer.decode(de_ids))
print("영어 텍스트:", decoder_tokenizer.decode(en_ids))

독일어 토큰화: [1962, 4705, 10, 1977, 9, 6921]
영어 토큰화: [302, 7, 450, 149, 740, 8, 4, 2558]
독일어 텍스트: Wiederaufnahme der Sitzungsperiode
영어 텍스트: Resumption of the session


In [ ]:
de_ids = encoder_tokenizer.encode(sample_de_text)
print(type(de_ids))
print(de_ids)
[1] + de_ids + [2]

<class 'list'>
[1962, 4705, 10, 1977, 9, 6921]


[1, 1962, 4705, 10, 1977, 9, 6921, 2]

In [ ]:
input_dim = len(encoder_tokenizer)
output_dim = len(decoder_tokenizer)
emb_dim = 256
hid_dim = 512
n_layers = 2
drop_ratio = 0.5
batch_size = 64
N_EPOCHS = 3
MAX_LEN = 50

In [ ]:
from torch.utils.data import DataLoader, Dataset

class TranslationDataset(Dataset):
    def __init__(self, data, encoder_tokenizer, decoder_tokenizer, max_len=MAX_LEN):
        self.data = data
        self.encoder_tokenizer = encoder_tokenizer
        self.decoder_tokenizer = decoder_tokenizer
        self.max_len = max_len
        self.pad_id = 0
        self.bos_id = 1
        self.eos_id = 2

    def __len__(self):
        return len(self.data['translation'])

    def __getitem__(self, idx):
        src_text = self.data['translation'][idx]['de']
        trg_text = self.data['translation'][idx]['en']

        src_ids = self.encoder_tokenizer.encode(src_text)
        trg_ids = self.decoder_tokenizer.encode(trg_text)

        src_ids = src_ids[:self.max_len]
        trg_input = [self.bos_id] + trg_ids[:self.max_len - 1]
        trg_label = trg_ids[:self.max_len - 1] + [self.eos_id]

        src_ids = src_ids + [self.pad_id] * (self.max_len - len(src_ids))
        trg_input = trg_input + [self.pad_id] * (self.max_len - len(trg_input))
        trg_label = trg_label + [self.pad_id] * (self.max_len - len(trg_label))

        return torch.tensor(src_ids), torch.tensor(trg_input), torch.tensor(trg_label)

train_data = TranslationDataset(dataset['train'], encoder_tokenizer, decoder_tokenizer, max_len=MAX_LEN)
validataion_data = TranslationDataset(dataset['validation'], encoder_tokenizer, decoder_tokenizer, max_len=MAX_LEN)
test_data = TranslationDataset(dataset['test'], encoder_tokenizer, decoder_tokenizer, max_len=MAX_LEN)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validataion_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

# Seq2Seq without Attention

In [ ]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout)

    def forward(self, x):
        x = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(x)
        return outputs, hidden, cell

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, n_layers):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, n_layers)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, hidden, cell):
        x = x.unsqueeze(0)
        x = self.embedding(x)
        x, (hidden, cell) = self.lstm(x, (hidden, cell))
        prediction = self.fc_out(x.squeeze(0))
        return prediction, hidden, cell

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg=None, is_train=True, max_len=50, sos_token = 1, eos_token=2):
        # 학습 모드에서는 trg_len 사용, 추론 모드에서는 max_len까지 동적 생성
        batch_size = src.shape[1]
        trg_vocab_size = self.decoder.fc_out.out_features
        outputs = []

        # 인코더를 통해 context 생성
        _, hidden, cell = self.encoder(src)

        if is_train:
            for t in range(0, trg.shape[0]):
                input = trg[t]
                output, hidden, cell = self.decoder(input, hidden, cell)
                outputs.append(output.unsqueeze(0))

        else:
		    # inference에서는 target(정답)이 없기 때문에 sos_token을 생성해줍니다.
            input = torch.full((batch_size,), sos_token, dtype=torch.long, device=self.device)
            finished = torch.zeros(batch_size, dtype=torch.bool, device=self.device)

            for t in range(max_len):
                output, hidden, cell = self.decoder(input, hidden, cell)
                outputs.append(output.unsqueeze(0))
                top1 = output.argmax(1)
                input = top1

                # 조기 종료 조건
                finished |= (top1 == eos_token)
                if finished.all():
                    break

        return outputs

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = Encoder(input_dim, emb_dim, hid_dim, n_layers, drop_ratio).to(device)
decoder = Decoder(output_dim, emb_dim, hid_dim, n_layers).to(device)
model = Seq2Seq(encoder, decoder, device).to(device)

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

In [ ]:
from tqdm import tqdm

def train(model, data_loader, optimizer, criterion):
    model.train()
    epoch_loss = 0

    for src, trg_input, trg_label in tqdm(data_loader):
        src, trg_input, trg_label = src.to(device), trg_input.to(device), trg_label.to(device)
        optimizer.zero_grad()

        outputs = model(src, trg_input, is_train=True)
        # print(len(outputs))
        # print(outputs[0].shape)
        # print(trg_label.shape)
        # (output_dim, batch_size, trg_len)을 (batch_size * trg_len, output_dim)으로 변환
        outputs = torch.cat(outputs, dim=0).view(-1, output_dim)
        trg_label = trg_label.view(-1)

        # print("Output shape:", outputs.shape) # (batch*length, vocab)
        # print("Target shape:", trg_label.shape) # (batch*length,)

        loss = criterion(outputs, trg_label)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(data_loader)

In [ ]:
for epoch in range(N_EPOCHS):
    train_loss = train(model, validation_loader, optimizer, criterion)
    print(f'Epoch {epoch+1}/{N_EPOCHS}, Train Loss: {train_loss:.4f}')

 36%|███▌      | 17/47 [00:37<01:03,  2.13s/it]Exception ignored in: <function tqdm.__del__ at 0x7e48f9abf490>
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/usr/local/lib/python3.10/dist-packages/tqdm/std.py", line 1267, in close
    if self.disable:
AttributeError: 'tqdm' object has no attribute 'disable'
100%|██████████| 47/47 [01:45<00:00,  2.23s/it]


Epoch 1/3, Train Loss: 7.0490


100%|██████████| 47/47 [01:43<00:00,  2.21s/it]


Epoch 2/3, Train Loss: 6.6504


100%|██████████| 47/47 [01:45<00:00,  2.24s/it]

Epoch 3/3, Train Loss: 6.4958


# Seq2Seq with Attention

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim)

    def forward(self, src):
        # src : (src_len, batch_size)
        embedded = self.embedding(src)  # embedded : (src_len, batch_size, emb_dim)
        outputs, (hidden, cell) = self.rnn(embedded)  # outputs : (src_len, batch_size, hidden_dim)

        return outputs, hidden, cell

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.W1 = nn.Linear(hidden_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: (batch_size, hidden_dim)
        # encoder_outputs: (src_len, batch_size, hidden_dim)

        src_len = encoder_outputs.shape[0]

        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)  # (batch_size, src_len, hidden_dim)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)  # (batch_size, src_len, hidden_dim)

        energy = torch.tanh(self.W1(encoder_outputs) + self.W2(hidden))  # (batch_size, src_len, hidden_dim)
        attention = self.v(energy).squeeze(2)  # (batch_size, src_len)

        return nn.functional.softmax(attention, dim=1)  # (batch_size, src_len)

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, attention):
        super(Decoder, self).__init__()

        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        # Decoder RNN에는 embedding만 입력
        self.rnn = nn.LSTM(emb_dim, hidden_dim)
        # 출력층에는 hidden state와 attention value가 결합되어 입력
        self.fc_out = nn.Linear(hidden_dim + hidden_dim, output_dim)

    def forward(self, input, hidden, cell, encoder_outputs):
        # input : (batch_size,)
        # hidden : (batch_size, hidden_dim)
        # encoder_outputs : (src_len, batch_size, hidden_dim)

        input = input.unsqueeze(0)  # input : (1, batch_size)
        embedded = self.embedding(input)  # embedded : (1, batch_size, emb_dim)

        # attention distribution을 계산합니다. decoder의 이전 hidden state, s_{t-1}와 encoder의 H가 입력됩니다.
        a = self.attention(hidden[-1], encoder_outputs)  # a : (batch_size, src_len)

        # H에 가중치를 부여해 attention value(Context vector) 계산
        a = a.unsqueeze(1)  # a : (batch_size, 1, src_len)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)  # encoder_outputs : (batch_size, src_len, hidden_dim)
        context = torch.bmm(a, encoder_outputs)  # context : (batch_size, 1, hidden_dim)
        context = context.permute(1, 0, 2)  # context : (1, batch_size, hidden_dim)

        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))

        # 출력층에서는 현재 hidden state와 context vector를 결합하여 예측값 생성
        output = output.squeeze(0)  # output : (batch_size, hidden_dim)
        context = context.squeeze(0)  # context : (batch_size, hidden_dim)
        prediction = self.fc_out(torch.cat((output, context), dim=1))  # (batch_size, output_dim)

        return prediction, hidden, cell

In [ ]:
class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        # src : (src_len, batch_size)
        # trg : (trg_len, batch_size)

        trg_len = trg.shape[0]
        batch_size = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        # 이전에는 동적으로 생성하기 위해 outputs = []를 사용했습니다.
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(device)

        # Encoder는 동일합니다.
        encoder_outputs, hidden, cell = self.encoder(src)

        for t in range(trg_len):
            input = trg[t]
            # 기존에는 이전 cell의 추론 값(input), hidden state(hidden, cell)만 입력했다면 encoder_outputs(H)가 추가됩니다.
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[t] = output

        # 인퍼런스 부분은 생략합니다.
        return outputs

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = Encoder(input_dim, emb_dim, hid_dim).to(device)
attention = BahdanauAttention(hid_dim).to(device)
decoder = Decoder(output_dim, emb_dim, hid_dim, attention).to(device)
model = Seq2SeqAttention(encoder, decoder).to(device)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

In [ ]:
def train(model, data_loader, optimizer, criterion):
    model.train()
    epoch_loss = 0

    for src, trg_input, trg_label in data_loader:
        src, trg_input, trg_label = src.to(device), trg_input.to(device), trg_label.to(device)
        optimizer.zero_grad()

        outputs = model(src, trg_input)

        # (trg_len, batch_size, output_dim)을 (batch_size * trg_len, output_dim)으로 변환
        outputs = outputs.view(-1, outputs.shape[-1])
        trg_label = trg_label.view(-1)

        loss = criterion(outputs, trg_label)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(data_loader)


In [ ]:
for epoch in range(N_EPOCHS):
    train_loss = train(model, validation_loader, optimizer, criterion)
    print(f'Epoch {epoch+1}/{N_EPOCHS}, Train Loss: {train_loss:.4f}')

Epoch 1/3, Train Loss: 7.0396
Epoch 2/3, Train Loss: 6.2677
Epoch 3/3, Train Loss: 5.9166
